# 04 - Joined analytical table 

Para responder a estas perguntas, precisamos de cruzar (JOIN) as quatro tabelas da camada Silver (`DEMO`, `DRUG`, `REAC` e `OUTC`).

⚠️ Atenção: No FAERS, um caso (primaryid) pode ter vários medicamentos (`DRUG`), várias reações (`REAC`) e vários desfechos (`OUTC`). Fazer um JOIN direto a todas as tabelas vai multiplicar as linhas (ex: 1 caso com 2 medicamentos e 3 reações = 6 linhas). Isto é o chamado fan-out.
Para garantir que as contagens ficam corretas, vamos usar sempre countDistinct("primaryid") em vez de um simples count() quando quisermos contar o número real de casos.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# 1. Definir o caminho onde as tabelas Silver foram guardadas
silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

# 2. Carregar as tabelas Delta para o dicionário silver_dfs
tables = ["demo", "drug", "reac", "outc"]
silver_dfs = {}

for table in tables:
    silver_dfs[table] = spark.read.format("delta").load(f"{silver_delta_path}/{table}")
    print(f"✅ Tabela {table.upper()} carregada com sucesso.")

# 3. Criar a Tabela Única (Gold Base Table) através de Joins
df_gold_base = (
    silver_dfs["demo"]
    .join(silver_dfs["drug"], on=["primaryid", "caseid"], how="inner")
    .join(silver_dfs["reac"], on=["primaryid", "caseid"], how="left")
    .join(silver_dfs["outc"], on=["primaryid", "caseid"], how="left")
)

display(df_gold_base.limit(5))

# Q1 - Top 10 medicamentos mais reportados e as reações adversas mais associadas a cada um

In [0]:
# 1. Encontrar os 10 medicamentos mais reportados
top_10_drugs = (
    df_gold_base
    .groupBy("drugname")
    .agg(F.countDistinct("primaryid").alias("total_casos"))
    .orderBy(F.desc("total_casos"))
    .limit(10)
)

# 2. Filtrar a base para os top 10 e contar as reações (pt) associadas
reactions_per_drug = (
    df_gold_base
    .join(top_10_drugs, on="drugname", how="inner")
    .filter(F.col("pt").isNotNull() & (F.col("pt") != "UNK"))
    .groupBy("drugname", "pt")
    .agg(F.countDistinct("primaryid").alias("reaction_count"))
)

# 3. Usar Window Function para extrair as top 3 reações por cada medicamento
w_q1 = Window.partitionBy("drugname").orderBy(F.desc("reaction_count"))

q1_result = (
    reactions_per_drug
    .withColumn("rank", F.row_number().over(w_q1))
    .filter(F.col("rank") <= 10)
    .orderBy("drugname", F.desc("reaction_count"))
)

display(q1_result)

# Q2 - Distribuição de eventos adversos por faixa etária

In [0]:
# Utilizar a coluna age_years normalizada na camada Silver
q2_result = (
    df_gold_base
    .filter(F.col("age_years").isNotNull())
    .withColumn(
        "age_group",
        F.when(F.col("age_years") < 18, "Pediatric")
         .when(F.col("age_years").between(18, 64), "Adult")
         .otherwise("Elderly")
    )
    .groupBy("age_group")
    .agg(F.countDistinct("primaryid").alias("total_events"))
    .withColumn(
        "percentage", 
        F.round((F.col("total_events") / F.sum("total_events").over(Window.partitionBy())) * 100, 2)
    )
    .orderBy(F.desc("total_events"))
)

display(q2_result)

# Q3 - Pares Medicamento-Reação com maior taxa de mortalidade

In [0]:
# 'DE' é o standard no FAERS para 'Death' (Morte) - Não tenho a certeza q esteja bem
q3_result = (
    df_gold_base
    .filter(F.col("drugname").isNotNull() & F.col("pt").isNotNull())
    .groupBy("drugname", "pt")
    .agg(
        F.countDistinct("primaryid").alias("total_cases"),
        F.countDistinct(F.when(F.col("outc_cod") == "DE", F.col("primaryid"))).alias("fatal_cases")
    )
    # Filtrar apenas pares com significância estatística (ex: > 100 casos reportados)
    .filter(F.col("total_cases") > 100)
    .withColumn("mortality_rate_pct", F.round((F.col("fatal_cases") / F.col("total_cases")) * 100, 2))
    .orderBy(F.desc("mortality_rate_pct"))
    .limit(20)
)

display(q3_result)

# Q4 - Padrões sazonais no reporte de eventos (por mês) - falta colocar por estação

In [0]:
q4_result = (
    df_gold_base
    .filter(F.col("event_dt").isNotNull())
    .withColumn("event_month", F.month("event_dt"))
    .groupBy("event_month")
    .agg(F.countDistinct("primaryid").alias("total_reports"))
    .orderBy("event_month")
)

display(q4_result)

# Q5 - Duração média da terapêutica por medicamento

⚠️ Atenção Técnica: Conforme o teu setup da camada Bronze (no ficheiro 01 - Initial setup...), o schema da tabela DRUG foi importado sem colunas de datas de início e fim terapêutico (geralmente drugstartdt e drugenddt no FAERS original).
Como não temos estes dados na camada Silver, não é possível calcular a aritmética de datas (Duração = Fim - Início).
Para resolveres isto no teu projeto, terás de voltar ao Notebook 01, adicionar essas duas colunas ao schema provisório da Bronze, normalizá-las no Notebook 02

# Q6 - Diferença de Outcomes nos Top 5 Medicamentos

In [0]:
# 1. Top 5 medicamentos
top_5_list = [row["drugname"] for row in top_10_drugs.limit(5).collect()]

# 2. Outcomes (DE=Death, HO=Hospitalization, LT=Life-Threatening, etc.)
q6_result = (
    df_gold_base
    .filter(F.col("drugname").isin(top_5_list))
    .filter(F.col("outc_cod").isNotNull() & (F.col("outc_cod") != "UNK"))
    .groupBy("drugname")
    .pivot("outc_cod")
    .agg(F.countDistinct("primaryid"))
    .fillna(0) # Preencher nulos 
)

display(q6_result)

# Q7 - Medicamentos com reportes crescentes ao longo do tempo (YoY Growth)

In [0]:
# Usamos a fda_dt (data em que a FDA recebeu o caso) para analisar tendências
reports_by_year = (
    df_gold_base
    .filter(F.col("fda_dt").isNotNull())
    .withColumn("report_year", F.year("fda_dt"))
    .groupBy("drugname", "report_year")
    .agg(F.countDistinct("primaryid").alias("yearly_reports"))
    .filter(F.col("yearly_reports") > 50) # Ignorar medicamentos com muito poucos casos
)

# Função Window (LAG) para comparar com o ano anterior
w_yoy = Window.partitionBy("drugname").orderBy("report_year")

q7_result = (
    reports_by_year
    .withColumn("prev_year_reports", F.lag("yearly_reports", 1).over(w_yoy))
    .filter(F.col("prev_year_reports").isNotNull())
    .withColumn(
        "yoy_growth_pct", 
        F.round(((F.col("yearly_reports") - F.col("prev_year_reports")) / F.col("prev_year_reports")) * 100, 2)
    )
    .orderBy(F.desc("yoy_growth_pct"))
)

display(q7_result)

# Q8 - Construir um "Drug Safety Score"